# Fetching Data

First we fetch the housing data from the url

In [ ]:
import os
import tarfile
from six.moves import urllib

DOWNLOAD_ROOT = "https://raw.githubusercontent.com/ageron/handson-ml2/master/"
HOUSING_PATH = os.path.join("datasets", "housing")
HOUSING_URL = DOWNLOAD_ROOT + "datasets/housing/housing.tgz"

def fetch_housing_data(housing_url=HOUSING_URL, housing_path=HOUSING_PATH):
    if not os.path.isdir(housing_path):
        os.makedirs(housing_path)
    tgz_path = os.path.join(housing_path, "housing.tgz")
    urllib.request.urlretrieve(housing_url, tgz_path)
    housing_tgz = tarfile.open(tgz_path)
    housing_tgz.extractall(path=housing_path)
    housing_tgz.close()

In [ ]:
fetch_housing_data()

After we fetched the data from the url we will write a function to extract the csv and turn it into a df. After this we will begin to explore the DF

In [ ]:
import pandas as pd

def load_housing_data(housing_path=HOUSING_PATH):
    csv_path = os.path.join(housing_path, "housing.csv")
    return pd.read_csv(csv_path)

In [ ]:
housing=load_housing_data()
housing.head()

In [ ]:
housing.info()

In [ ]:
housing.ocean_proximity.value_counts()

In [ ]:
housing.describe()

In [ ]:
# only in a Jupyter notebook
%matplotlib inline   

import matplotlib.pyplot as plt

housing.hist(bins=50, figsize=(20,15))
plt.show()

Notice values over 500,000 are caped in median_house_value (the target data). This means the model might not work properly with values over this range. Let's look at the data closer where values are over 500,000.

In [ ]:
housing.loc[housing['median_house_value']>500000].hist(figsize=(20,15)) #(bins=50, figsize=(20,15))
plt.show()

In [ ]:
housing['median_house_value'].loc[housing['median_house_value']>500000].value_counts()

### Test set

Before doing any changes to our data we must set apart a testing set. Let's do this now before messing with the data and wrongly overfitting it.

In [ ]:
#train test split function with numpy permutating indices from an np array
import numpy as np

def split_train_test(data, test_ratio):
    np.random.seed(42) #notice this is The Answer to the Ultimate Question of Life, the Universe, and Everything
    shuffled_indices = np.random.permutation(len(data))
    test_set_size = int(len(data) * test_ratio)
    test_indices = shuffled_indices[:test_set_size]
    train_indices = shuffled_indices[test_set_size:]
    return data.iloc[train_indices], data.iloc[test_indices]

In [ ]:
train_set, test_set = split_train_test(housing, 0.2)
len(train_set)

In [ ]:
len(test_set)

In [ ]:
#lets see if the indexes are mismatched now
test_set.index

this can also be done with sklearn

In [ ]:
from sklearn.model_selection import train_test_split

train_set, test_set = train_test_split(housing, test_size=0.2, random_state=42)

Note: since we are taking our data from online source it would be a good idea toset an identifier to our current data. This will secure that our model doesnt change with a new data update. This could be done if the data has an identifier. If this is not possible we could set an index to current data and concatenate new data. Other possibility is ussing a mix of unchanging data variables such as longitude and latitude.

#### Categorical median income

The following classifies different income strata

In [ ]:
housing["income_cat"] = pd.cut(housing["median_income"],bins=[0., 1.5, 3.0, 4.5, 6., np.inf],labels=[1, 2, 3, 4, 5])

In [ ]:
housing["income_cat"].hist()

with this we can run a stratified sampling

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit

split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_index, test_index in split.split(housing, housing["income_cat"]):
    strat_train_set = housing.loc[train_index]
    strat_test_set = housing.loc[test_index]

In [ ]:
# lets check if it worked
strat_test_set["income_cat"].value_counts() / len(strat_test_set)

Let's remove the income feature we came up with

In [ ]:
for set_ in (strat_train_set, strat_test_set):
    set_.drop("income_cat", axis=1, inplace=True)

Let's make a copy of the training set so we dont alter it while we play with it

In [ ]:
housing = strat_train_set.copy()

### Visualize the Data

We will begin to plot the data in order to understand our data better 

In [ ]:
housing.plot(kind="scatter", x="longitude", y="latitude") #easy x, y axis assignation and scatterplot
plt.show() #remember this line is not always necessary

Yep, thats California all right. But now what? Lets set an alpha to visualize density

In [ ]:
housing.plot(kind="scatter", x="longitude", y="latitude", alpha=0.1)
plt.show()

We can now dilusidate the bay area and LA and San diego. We can do better than that

In [ ]:
housing.plot(kind="scatter", x="longitude", y="latitude", #alpha=0.4,
             s=housing["population"]/100, #size given by population. Over 100 makes it 35 as max value
             label="population", #label on the right top
             figsize=(13,8),
             c="median_house_value", #color given by median h value
             cmap=plt.get_cmap("jet"), #this sets low values to blue and high values to red
             colorbar=True, #add a colorbar of what the color values represent
)
plt.legend()
plt.show()

### Correlations

Now let's look for correlations. We can start with the Pearson's r

In [ ]:
corr_matrix = housing.iloc[:,:-1].corr()

In [ ]:
# Plot the heatmap
import seaborn as sns

plt.figure(figsize=(8, 6)) # Set the figure size
sns.heatmap(
    corr_matrix, 
    annot=True,          # Add the correlation values as text annotations
    cmap='coolwarm',     # Use a diverging color palette (e.g., blue for negative, red for positive)
    vmin=-1,             # Ensure the color scale spans from -1 to 1
    vmax=1,
    center=0,            # Center the color gradient at 0 (no correlation)
    fmt=".2f",           # Format the annotations to two decimal places
    linewidths=0.5,      # Add slight lines between cells for clarity
    square=True          # Ensure cells are square
)

plt.title('Pearson Correlation Heatmap')
plt.show()

Lets put special emphasis to our target feature so we understand our values better

In [ ]:
corr_matrix["median_house_value"].sort_values(ascending=False)

It is clear that the only significantly linearly correlated feature is the median income, which we kinda already knew. Other than that longitude and latittude are slightly correlated negatively which means that values tend to go lower as we go north or east.

Just in case, we will make a matrix scatter plotted just to spot if there is a polinomial or any other type of correlation

In [ ]:
from pandas.plotting import scatter_matrix

attributes = ["median_house_value", "median_income", "total_rooms",
"housing_median_age"]
scatter_matrix(housing[attributes], figsize=(12, 8))
plt.show()

This confirms our previous suspicions. Lets see that median income up close

In [ ]:
housing.plot(kind='scatter',y="median_house_value", x="median_income", alpha=0.5, figsize=(12, 8))
plt.show()

We can see a horizontal line at  $450,000$, another around $350,000$, perhaps one around $280,000$, and a few more below that. It's probably a good idea to get rid of them before the training. Also always remember to ask before removing things.

### Feature engineering

Notice the total number of rooms in a district is not very useful if you don’t know how many
households there are. What you really want is the **number_of_rooms_per_household**.
Similarly, the **total_number_of_bedrooms** by itself is not very useful: you probably
want to **compare** it to the **number_of_rooms**. And the **population_per_household** also
seems like an interesting attribute combination to look at. Let’s create these new
attributes.

In [ ]:
housing["rooms_per_household"] = housing["total_rooms"]/housing["households"]
housing["bedrooms_per_room"] = housing["total_bedrooms"]/housing["total_rooms"]
housing["population_per_household"]=housing["population"]/housing["households"]

In [ ]:
housing.head(3)

With these **new values** lets run a **correlation** again

In [ ]:
#column selection without cathegorical variables
tam_min=housing.shape[1]-4 #valor antes del la columna categorica
tam_max=housing.shape[1]-3 #valor después de la columna categorica
corr_matrix = housing.iloc[:,np.r_[:tam_min,tam_max:13]].corr()

In [ ]:
plt.figure(figsize=(10,8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1, center=0,
            fmt="0.2f", linewidth=0.5)
plt.title=('Pearson Correlation Heatmap')
plt.show()

In [ ]:
corr_matrix["median_house_value"].sort_values(ascending=False)

Not bad! These featured engineering gave us a **better correlation** than the values all by **themselves**.

## Prepare data for ML

Time to prepare the data for a ML algorithm. Make sure you write functions since this will make the job easier to work on production. Let's start by reverting our changes.

In [ ]:
housing = strat_train_set.drop("median_house_value", axis=1)
housing_labels = strat_train_set["median_house_value"].copy()

#### Dealing with NAN data

Remember we have the feature **total_bedrooms** that has nan values we must deal with.

In [ ]:
housing.isna().sum() #count nan values

Easier methods to deal with NAN values are: \
**Option 1:** Drop NAN values. \
**Option 2:** Drop the whole feature. \
**Option 3:** Replace NAN values with the median. 

In [ ]:
#housing.dropna(subset=["total_bedrooms"])                                #option 1

#housing.drop("total_bedrooms", axis=1)                                   #option 2

median = housing["total_bedrooms"].median()                               #option 3
#housing["total_bedrooms"].fillna(median, inplace=True)

In [ ]:
median

Option 3 can be done with an Scikit-Learn simple imputer function that makes the calculation of the median of all numerical features

In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

In [ ]:
housing_num = housing.drop("ocean_proximity", axis=1) #df with only numerical values

In [ ]:
imputer.fit(housing_num) #applying the method

In [ ]:
imputer.statistics_ #sklearn output

In [ ]:
housing_num.median().values #simple median function

As you can see we get the **same median value** we would get with the traditional function. However it is **easier to input data** with sklearn like this:

In [ ]:
X = imputer.transform(housing_num) #inputed median array

In [ ]:
ousing_tr = pd.DataFrame(X, columns=housing_num.columns) #transformed inputed array to df 

#### Handling categorical values

Most ML algorithms rather work with numbers instead of categories so we'll use Scikit-Learn’s OrdinalEncoder class

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

ordinal_encoder = OrdinalEncoder() #ordinal encoder class

In [ ]:
housing_cat = housing[["ocean_proximity"]] #ocean proximity column. only this categorical value

housing_cat_encoded = ordinal_encoder.fit_transform(housing_cat) 
housing_cat_encoded[:10]

You can also see the original values of each category in an array like this

In [ ]:
ordinal_encoder.categories_

The integer values 0-4 might confuse the ML algoritm assigning some type of sense of sentiument analyisis to the problem. Because of this we might rather use one-hot-encoding

In [ ]:
from sklearn.preprocessing import OneHotEncoder

cat_encoder= OneHotEncoder()
housing_cat_1hot=cat_encoder.fit_transform(housing_cat)
housing_cat_1hot

To save memory 1 hot encoding only saves the postions of the 1's. This is called a sparsed matrix. Howerver this can be tranformed to a regular array like this

In [ ]:
housing_cat_1hot.toarray()

And again, you can also see the original values of each category in an array like this

In [ ]:
cat_encoder.categories_

Note: With 1 hot encoding too many categories can slow down training and degrade performance. Be wary

### Custom transformers

Here is a small transformer class that adds
the combined attributes we discussed earlier:

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

rooms_ix, bedrooms_ix, population_ix, households_ix = 3, 4, 5, 6

class CombinedAttributesAdder(BaseEstimator, TransformerMixin):
    def __init__(self, add_bedrooms_per_room = True): # no *args or **kargs
        self.add_bedrooms_per_room = add_bedrooms_per_room
    def fit(self, X, y=None):
        #print("Fitting CombinedAttributesAdder")
        self.sklearn_sucks_ = True
        return self  # nothing else to do
    def transform(self, X, y=None):
        rooms_per_household = X[:, rooms_ix] / X[:, households_ix]
        population_per_household = X[:, population_ix] / X[:, households_ix]
        if self.add_bedrooms_per_room:
            bedrooms_per_room = X[:, bedrooms_ix] / X[:, rooms_ix]
            return np.c_[X, rooms_per_household, population_per_household, bedrooms_per_room]
        else:
            return np.c_[X, rooms_per_household, population_per_household]

In [ ]:
attr_adder = CombinedAttributesAdder(add_bedrooms_per_room=False)
housing_extra_attribs = attr_adder.transform(housing.values)

It is a good practice to create a class of all the cleaning you would like to add using the scikit transform method. In this case it is a feature engineer method

#### Feature Scaling

To do feature scaling we could either normalize (make a bell curve out of values from 0 to 1. Recommended for neural networks) the data or standarize it (not recommended for neural networks). Must remember that standarization is not sensitive to outliers. For now we will use a StandardScaler for our data cleaning pipeline.  

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

num_pipeline=Pipeline([
    ('imputer', SimpleImputer(strategy="median")),
    ('attribs_adder', CombinedAttributesAdder()),
    ('std_scaler', StandardScaler())
])

In [ ]:
housing_num_tr= num_pipeline.fit_transform(housing_num)

What about 1 hot encoding and categorical data? Not to worry we can use a full pipeline with the ColumnTransformer class and get all the data cleaned.

In [ ]:
from sklearn.compose import ColumnTransformer

num_attribs=list(housing_num) #list of numerical features
cat_attribs=['ocean_proximity']

full_pipeline= ColumnTransformer([
    ('num', num_pipeline, num_attribs),
    ('cat', OneHotEncoder(), cat_attribs)
])

In [ ]:
housing_prepared=full_pipeline.fit_transform(housing)

### ML Time!

Let's start with a simple linear regression model

In [ ]:
from sklearn.linear_model import LinearRegression

lin_reg=LinearRegression()
lin_reg.fit(housing_prepared, housing_labels)

In [ ]:
#lets look at the results in the model

some_data = housing.iloc[:5]
some_labels = housing_labels.iloc[:5]
some_data_prepared = full_pipeline.transform(some_data)
print("Predictions:", lin_reg.predict(some_data_prepared))

In [ ]:
print("Labels:", list(some_labels))

We can see the predictions work but they're not accuarate. Let's meassure the error. In this case we'll use RMSE (root mean square error). Don't forget that the RMSE is given by the following formula: 

\begin{equation}
RMSE(\mathbf{X}, h)= \sqrt{\frac{1}{m}\sum_{i=1}^{m} (h(\mathbf{x}^{(i)})-y^{(i)})^2}
\end{equation}

Where $\mathbf{X}$ is a matrix, $m$ is the number of instances in the dataset, $\mathbf{x}^{(i)}$ is a vector of all the feature values excluding the label (a subset of $\mathbf{X}$), $h$ is the ML prediction also called hypothesis and $y^{(i)}$ is the label.

RMSE is generally the preferred performance measure for regression tasks.

Also remember that there are other ways to meassure error such as the Mean Absolute Error (MAE) given by the following formula:

\begin{equation}
MAE(\mathbf{X}, h)= \frac{1}{m}\sum_{i=1}^{m} |h(\mathbf{x}^{(i)})-y^{(i)}|
\end{equation}

Remember that the RMSE is more sensitive to outliers than the MAE. This is also why RMSE generally performs better in bell shaped curves since outliers are exponencially rare in this curves. For a better explanation of why this happens in a mathematical way focused in norms refer to page 44 of Hands on ML with Scikit-Learn, Keras and Tensorflow 2nd edition.

In [ ]:
from sklearn.metrics import mean_squared_error

housing_predictions= lin_reg.predict(housing_prepared)
lin_mse=mean_squared_error(housing_labels, housing_predictions)
lin_rmse=np.sqrt(lin_mse)
lin_rmse

Okay this model is underfitted. Our options are:
* Select a more powerful model
* Feed the algorithm with better features
* Reduce the constraints on the model (parameters or regularization)

So we'll use a more powerful model. we'll use a DecisionTreeRegressor which is capable of finding nonlinear relationships in the data.

In [ ]:
from sklearn.tree import DecisionTreeRegressor

tree_reg= DecisionTreeRegressor()
tree_reg.fit(housing_prepared, housing_labels)

In [ ]:
#Lets evaluate the model

housing_predictions=tree_reg.predict(housing_prepared)
tree_mse=mean_squared_error(housing_labels, housing_predictions)
tree_rmse=np.sqrt(tree_mse)
tree_rmse

Well, we finally did it. We overfitted the data.

### Cross Validation

To evaluate our models we will use cross validation.

In [ ]:
#10 fold cross validation for a regression tree
from sklearn.model_selection import cross_val_score

scores = cross_val_score(tree_reg, housing_prepared, housing_labels,scoring="neg_mean_squared_error", cv=10)
tree_rmse_scores = np.sqrt(-scores)

In [ ]:
def display_scores(scores):
    print("Scores:", scores)
    print("Mean:", scores.mean())
    print("Standard deviation:", scores.std())

In [ ]:
display_scores(tree_rmse_scores)

The Decision Tree has a score of approximately 71,407, generally ±2,439. You would not have this information if you just used one validation set. But cross-validation comes at the cost of training the model several times, so it is not always possible. Now let's compute cross validation but for a regular Linear regression.

In [ ]:
lin_scores = cross_val_score(lin_reg, housing_prepared, housing_labels, scoring="neg_mean_squared_error", cv=10)
lin_rmse_scores = np.sqrt(-lin_scores)
display_scores(lin_rmse_scores)

That’s right: the Decision Tree model is overfitting so badly that it performs worse than the Linear Regression model. Let's now try a RandomForestRegressor

In [ ]:
from sklearn.ensemble import RandomForestRegressor

forest_reg = RandomForestRegressor()
forest_reg.fit(housing_prepared, housing_labels)

In [ ]:
# RMSE of Random Forest
housing_predictions=forest_reg.predict(housing_prepared)
forest_mse=mean_squared_error(housing_labels, housing_predictions)
forest_rmse=np.sqrt(forest_mse)
forest_rmse

In [ ]:
forest_scores = cross_val_score(forest_reg, housing_prepared, housing_labels, scoring="neg_mean_squared_error", cv=10)
forest_rmse_scores = np.sqrt(-forest_scores)
display_scores(forest_rmse_scores)

as you can see this is how you compare each model to see how well they perform. You can also save your finished models with pythons pickle module like this:

In [ ]:
#from sklearn.externals import joblib
#joblib.dump(my_model, "my_model.pkl")
# and later...
#my_model_loaded = joblib.load("my_model.pkl")

#### Gridsearch

This lets you itterate all possible parameter combinations. (This code will be commented to avoid long times when running all the cells in the notebook)

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
#RANDOM FOREST GRIDSEARCH

# param_grid = [
#     {'n_estimators': [3, 10, 30], 'max_features': [2, 4, 6, 8]},
#     {'bootstrap': [False], 'n_estimators': [3, 10], 'max_features': [2, 3, 4]},
# ]

# forest_reg = RandomForestRegressor()

# grid_search = GridSearchCV(forest_reg, param_grid, cv=5,
#                            scoring='neg_mean_squared_error',
#                            return_train_score=True)

In [ ]:
# grid_search.fit(housing_prepared, housing_labels)

In [ ]:
# grid_search.best_params_

6 and 30 are the highest values so we should try with bigger values. We'll do it later...

In [ ]:
# grid_search.best_estimator_ #also it is possible to get the best estimator like this

Also you can see the evaluation scores like this...

In [ ]:
# cvres = grid_search.cv_results_
# for mean_score, params in zip(cvres["mean_test_score"], cvres["params"]):
#     print(np.sqrt(-mean_score), params)

#### Feature importance

It is also possible to get the feature importance with gridsearch

In [ ]:
#feature_importances = grid_search.best_estimator_.feature_importances_

In [ ]:
#feature_importances

In [ ]:
#let's display them with their attributing names

# extra_attribs = ["rooms_per_hhold", "pop_per_hhold", "bedrooms_per_room"]
# cat_encoder = full_pipeline.named_transformers_["cat"]
# cat_one_hot_attribs = list(cat_encoder.categories_[0])
# attributes = num_attribs + extra_attribs + cat_one_hot_attribs
# sorted(zip(feature_importances, attributes), reverse=True)

Here will resume what we promised we'll do later and also we'll do the exercises in chapter 2 of hands on ML

Here is what we promised we'll do later, which is adding higher values in the grid search for the random forest.

In [ ]:
#RANDOM FOREST GRIDSEARCH FOR HIGHER VLAUES THAN 30 OR 6

# param_grid = [
#     {'n_estimators': [30, 50,100,150,250, 300], 'max_features': [2, 4, 6, 8,15,20,50,100]},
#     {'bootstrap': [False], 'n_estimators': [3, 10], 'max_features': [2, 3, 4]},
# ]

# forest_reg = RandomForestRegressor()

# grid_search = GridSearchCV(forest_reg, param_grid, cv=5,
#                            scoring='neg_mean_squared_error',
#                            return_train_score=True)

In [ ]:
#grid_search.fit(housing_prepared, housing_labels)

In [ ]:
#grid_search.best_params_

Ok now I know for sure 6 is a good parameter (like there was a reason to doubt) and that the ideal n_estimator parameter is below 300 and probably around 225. I'lll keep exeprimenting until I get a good ideal parameter.

In [ ]:
#FOCUSED RANDOM FOREST GRID SEARCH

# param_grid = [
#     {'n_estimators': [290,285, 280], 'max_features': [6]},
#    # {'bootstrap': [False], 'n_estimators': [3, 10], 'max_features': [2, 3, 4]},
# ]

# forest_reg = RandomForestRegressor()

# grid_search = GridSearchCV(forest_reg, param_grid, cv=5,
#                            scoring='neg_mean_squared_error',
#                            return_train_score=True)

In [ ]:
#grid_search.fit(housing_prepared, housing_labels)

In [ ]:
#grid_search.best_params_

In [ ]:
#features importance

#feature_importances =grid_search.best_estimator_.feature_importances_

In [ ]:
#let's display them with their attributing names

# extra_attribs = ["rooms_per_hhold", "pop_per_hhold", "bedrooms_per_room"]
# cat_encoder = full_pipeline.named_transformers_["cat"]
# cat_one_hot_attribs = list(cat_encoder.categories_[0])
# attributes = num_attribs + extra_attribs + cat_one_hot_attribs
# sorted(zip(feature_importances, attributes), reverse=True)

Let's test the random forest final model

In [ ]:
# final_forest_model = grid_search.best_estimator_
# X_test = strat_test_set.drop("median_house_value", axis=1)
# y_test = strat_test_set["median_house_value"].copy()
# X_test_prepared = full_pipeline.transform(X_test)
# final_predictions = final_forest_model.predict(X_test_prepared)
# final_mse = mean_squared_error(y_test, final_predictions)
# final_rmse = np.sqrt(final_mse)   

In [ ]:
#final_rmse

But how do I know this is better than the previous model? Should I launch it?. You can check wether this is actually worth it or not by computing a confidence interval with scipy.stats.t.interval(). There is an example in the book on page 83. Maybe we'll test the actual final model in the end.

### Book problems 

It's time to start dealing with the book problems.


1. Try a Support Vector Machine regressor (sklearn.svm.SVR), with various hyper
parameters such as kernel="linear" (with various values for the C hyperparameter) or kernel="rbf" (with various values for the C and gamma
hyperparameters). Don’t worry about what these hyperparameters mean for now.
How does the best SVR predictor perform?

In [ ]:
#Here we will run it without the grid search params just for the sake of understanding the main model
from sklearn.svm import SVR


svm =SVR(kernel='linear')
svm.fit(housing_prepared, housing_labels)

In [ ]:
#Lets evaluate the model

svm_predictions=svm.predict(housing_prepared)
svm_mse=mean_squared_error(housing_labels, svm_predictions)
svm_rmse=np.sqrt(svm_mse)
svm_rmse

Now that we ran the basic svm let's start running all the requisites from the problem, but we'll make sure we only get the most efficient parameters with the gridsearch option. I am familiarized with the kernels so I won't worry about it now. 

Up to this moment we will need an alarm that tells us when the code is ready. We will use winsound for this and we will add a code that will beep at the end

In [ ]:
import winsound
#import time

Now let's start making a gridsearch for the svm

In [ ]:
# kernel="linear" (with various values for the C hyperparameter)
#or kernel="rbf" (with various values for the C and gamma hyperparameters)


# param_grid = [
#     {'kernel':['linear'],'C':[1.0, 1.5 ,10,50 ] },
#     {'kernel':['rbf'], 'C':[1.0, 1.5 ,10,50 ], 'gamma': ['scale', 3, 5, 10]}
# ]

# svm =SVR()

# grid_search = GridSearchCV(svm, param_grid, cv=5,
#                            scoring='neg_mean_squared_error',
#                            return_train_score=True)

In [ ]:
#grid_search.fit(housing_prepared, housing_labels)

In [ ]:
#grid_search.best_params_

50 means it is topped out and we might need a bigger c. But training takes so long that I won't botter for now to try with bigger C vlaues. Let's evaluate this model.

In [ ]:
# svm_model = grid_search.best_estimator_

# svm_predictions = svm_model.predict(X_test_prepared)
# svm_mse = mean_squared_error(y_test, svm_predictions)
# svm_rmse = np.sqrt(svm_mse)  

# svm_rmse

Apparently a random model works better this time. Let's move on with the second problem.

2. Try replacing GridSearchCV with RandomizedSearchCV.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, randint

In [ ]:
# Define parameter distributions

# param_distributions = [
#     {'kernel': ['linear'],                  # linear kernel – no gamma needed
#      'C': uniform(loc=50, scale=150)},        # sample C uniformly between 1 and 51? Actually loc=1, scale=50 gives range [1, 51). You can adjust.
#     {'kernel': ['rbf'],
#      'C': uniform(loc=1, scale=50),         # same uniform for C
#      'gamma': uniform(loc=1, scale=50)}          # gamma as categorical choices
# ]

In [ ]:
# SVM RANDOM SEARCH

# random_search = RandomizedSearchCV(
#     svm,
#     param_distributions,
#     n_iter=20,               # try 20 random combinations
#     cv=5,
#     scoring='neg_mean_squared_error',
#     return_train_score=True,
#     random_state=42          # for reproducibility
# )

In [ ]:
#random_search.fit(housing_prepared, housing_labels)

In [ ]:
# print("Best params:", random_search.best_params_)
# print("Best score (neg MSE):", random_search.best_score_)

In [ ]:
# Use best estimator

# svm_model = random_search.best_estimator_
# svm_predictions = svm_model.predict(X_test_prepared)
# svm_mse = mean_squared_error(y_test, svm_predictions)
# svm_rmse = np.sqrt(svm_mse)
# print("RMSE on test set:", svm_rmse)

3. Try adding a transformer in the preparation pipeline to select only the most
important attributes.

In [ ]:
forest_reg = RandomForestRegressor(max_features=6, n_estimators=280)

In [ ]:
forest_reg.fit(housing_prepared, housing_labels)

In [ ]:
forest_predictions=forest_reg.predict(housing_prepared)
forest_mse=mean_squared_error(housing_labels, forest_predictions)
forest_rmse=np.sqrt(forest_mse)
forest_rmse

In [ ]:
forest_reg.feature_importances_

In [ ]:
#let's display them with their attributing names

feature_importances=forest_reg.feature_importances_

extra_attribs = ["rooms_per_hhold", "pop_per_hhold", "bedrooms_per_room"]
cat_encoder = full_pipeline.named_transformers_["cat"]
cat_one_hot_attribs = list(cat_encoder.categories_[0])
attributes = num_attribs + extra_attribs + cat_one_hot_attribs
sorted(zip(feature_importances, attributes), reverse=True)

So with the best random forest parameters we still have the same feature importances. It's time to add this to the pipeline. We will first make a class and then we will add it to the pipeline. Also we must make clear that since feature importance is only available for random forest. This pipeline is only meant to be used for random forest.

In [ ]:
#Best features for numerical columns only.  It's not deling with cathegorical columns
#Class could be improved with feature importances automatic float filter. 
#In this case we are only manually taking feature columns with values over 0.06  


class BestFeatures(BaseEstimator, TransformerMixin):
    def __init__(self, activate_best=True, best_indices=None):
        self.activate_best = activate_best
        self.best_indices = best_indices

    def fit(self, X, y=None):
        self.sklearn_sucks_ = True
        return self

    def transform(self, X):
        X = np.asarray(X)
        if not self.activate_best:
            return X
        
        indices = self.best_indices
        if indices is None:
            indices = [0, 1, 7, 8, 9, 10]

        return X[:, indices]

In [ ]:
forest_num_pipeline=Pipeline([
    ('imputer', SimpleImputer(strategy="median")),
    ('attribs_adder', CombinedAttributesAdder()),
    ('std_scaler', StandardScaler()),
    ('best_feature_importance',BestFeatures(activate_best=True) )
])

In [ ]:
forest_full_pipeline= ColumnTransformer([
    ('num', forest_num_pipeline, num_attribs),
    ('cat', OneHotEncoder(), cat_attribs)
])

In [ ]:
forest_housing_prepared=forest_full_pipeline.fit_transform(housing)

In [ ]:
forest_reg.fit(forest_housing_prepared, housing_labels)

In [ ]:
#Random forest RMSE validation

forest_predictions=forest_reg.predict(forest_housing_prepared)
forest_mse=mean_squared_error(housing_labels, forest_predictions)
forest_rmse=np.sqrt(forest_mse)
forest_rmse

In [ ]:
#Optimized random forest crossed validation

forest_scores = cross_val_score(forest_reg, forest_housing_prepared, housing_labels, scoring="neg_mean_squared_error", cv=10)
forest_rmse_scores = np.sqrt(-forest_scores)
display_scores(forest_rmse_scores)

This random forest model seems like is the best model so far with a validation RMSE of *18,043* and a cross validation mean RMSE of *49,026*. This might be due to slight overfitting which might need some prunning later on. To check this we will need to use the test dataset in the end.

4. Try creating a single pipeline that does the full data preparation plus the final
prediction

In [ ]:
#we will rewrite previous pipelines to understand what is going on in the final pipeline

forest_num_pipeline=Pipeline([
    ('imputer', SimpleImputer(strategy="median")),
    ('attribs_adder', CombinedAttributesAdder()),
    ('std_scaler', StandardScaler()),
   ('best_feature_importance',BestFeatures(activate_best=True) )
])

In [ ]:
#preprocessing column transformer

forest_preprocessing_pipeline= ColumnTransformer([
    ('num', forest_num_pipeline, num_attribs),
    ('cat', OneHotEncoder(), cat_attribs)
])

In [ ]:
#Final pipeline

final_pipeline= Pipeline([
    ('preprocessing', forest_preprocessing_pipeline),
    ('model_regressor',RandomForestRegressor(max_features=6, n_estimators=280))
    ])

In [ ]:
final_pipeline.fit(housing,housing_labels)

In [ ]:
# Predict with the pipeline
pipeline_predictions = final_pipeline.predict(strat_test_set.drop(['median_house_value'],axis=1))

In [ ]:
#Random forest RMSE validation

pipeline_mse=mean_squared_error(strat_test_set.loc[:,'median_house_value'], pipeline_predictions)
pipeline_rmse=np.sqrt(pipeline_mse)
pipeline_rmse

What we learned from this process? Basically that sklearn sucks. We had to add a ```self.sklearn_sucks_ = True``` trash variable in our fit functions in order for the code to take our custom transformers as valid when fitting. So in our final pipeline we got a RMSE of *46950.8* which is not bad but could be improved. Next We'll stick the chatgpt solution just for future reference:

In [ ]:
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin

# indices (keep same as before)
rooms_ix, bedrooms_ix, population_ix, households_ix = 3, 4, 5, 6

class CombinedAttributesAdder(BaseEstimator, TransformerMixin):
    def __init__(self, add_bedrooms_per_room=True):
        self.add_bedrooms_per_room = add_bedrooms_per_room

    def fit(self, X, y=None):
        X = np.asarray(X)
        # set a fitted attribute (anything ending with "_")
        # also set n_features_in_ which sklearn commonly uses
        try:
            self.n_features_in_ = X.shape[1]
        except Exception:
            self.n_features_in_ = None
        self.is_fitted_ = True
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float)
        rooms = X[:, rooms_ix]
        bedrooms = X[:, bedrooms_ix]
        population = X[:, population_ix]
        households = X[:, households_ix]

        # guards to avoid division by zero
        households_safe = np.where(households == 0, 1.0, households)
        rooms_safe = np.where(rooms == 0, 1.0, rooms)

        rooms_per_household = rooms / households_safe
        population_per_household = population / households_safe

        if self.add_bedrooms_per_room:
            bedrooms_per_room = bedrooms / rooms_safe
            out = np.c_[X, rooms_per_household, population_per_household, bedrooms_per_room]
        else:
            out = np.c_[X, rooms_per_household, population_per_household]
        return out


class BestFeatures(BaseEstimator, TransformerMixin):
    def __init__(self, activate_best=True, best_indices=None):
        self.activate_best = activate_best
        self.best_indices = best_indices

    def fit(self, X, y=None):
        X = np.asarray(X)
        # record input shape and mark as fitted
        try:
            self.n_features_in_ = X.shape[1]
        except Exception:
            self.n_features_in_ = None
        self.is_fitted_ = True
        return self

    def transform(self, X):
        X = np.asarray(X)
        if not self.activate_best:
            return X
        indices = self.best_indices if self.best_indices is not None else [0, 1, 7, 8, 9, 10]
        return X[:, indices]

In [ ]:
# 1) re-create numeric pipeline, ColumnTransformer, and final_pipeline
forest_num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy="median")),
    ('attribs_adder', CombinedAttributesAdder()),
    ('std_scaler', StandardScaler()),
    ('best_feature_importance', BestFeatures(activate_best=True))
])

forest_preprocessing_pipeline = ColumnTransformer([
    ('num', forest_num_pipeline, num_attribs),
    ('cat', OneHotEncoder(), cat_attribs)
], n_jobs=1)   # debug-friendly: force serial execution

final_pipeline = Pipeline([
    ('preprocessing', forest_preprocessing_pipeline),
    ('model_regressor', RandomForestRegressor(max_features=6, n_estimators=280))
])

# 2) fit
final_pipeline.fit(housing, housing_labels)

# 3) predict
X_test = strat_test_set.drop(['median_house_value'], axis=1)
preds = final_pipeline.predict(X_test)
print("Predictions shape:", preds.shape)

In [ ]:
#Random forest RMSE validation

pipeline_mse=mean_squared_error(strat_test_set.loc[:,'median_house_value'], preds)
pipeline_rmse=np.sqrt(pipeline_mse)
pipeline_rmse

In [ ]:
preds

5. Automatically explore some preparation options using GridSearchCV.

In [ ]:
#alert sound code

duration = 4000  # milliseconds 
freq = 440       # Hz


winsound.Beep(freq, duration)
print("Code is ready! Beep played.")